# Lab 2 — Routing: classify once, then hand off to a specialist (§19.1.2)

**Routing** = a classifier (an LLM or a traditional model) looks at the input first, then sends it to a **specialist handler**

The Cogito Learn teaching assistant gets 3 types of questions:

| Type | Sent to |
|---|---|
| 📘 `concept` concept | a prompt that's good at explaining theory |
| 🐞 `code_error` code error | a prompt with a code-running tool |
| 📅 `admin` admin (exam dates, etc.) | a small, cheap model |

**Use when:** the task has several clearly distinct types, and each type needs a different prompt, tool, or model

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.1.2 (p.370) — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`19-1-lab2-routing.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up client + ask_llm ----
import os, json
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}

def ask_llm(prompt, temperature=0.7, schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}}} if schema else {}
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA, temperature=temperature,
                                          messages=[{"role": "user", "content": prompt}], **kw)
    text = resp.choices[0].message.content
    return json.loads(text) if schema else text

## 1. Student questions

6 simulated questions, each with an `expect` field (the correct answer) for checking the router — the router **never sees** this field

In [ ]:
QUESTIONS = [
    {"q": "How is ReAct different from prompt chaining?", "expect": "concept"},
    {"q": "When is the midterm exam, and in which room?", "expect": "admin"},
    {"q": "I want to see my lab3 score, but running this gives an error",
     "code": 'scores = {"lab1": 8, "lab2": 9}\nprint(scores["lab3"])', "expect": "code_error"},
    {"q": "Can I submit a lab late? How many points are deducted?", "expect": "admin"},
    {"q": "Why does my average function break on an empty list?",
     "code": "def average(xs):\n    return sum(xs) / len(xs)\nprint(average([]))", "expect": "code_error"},
    {"q": "Does an LLM hallucination count as a kind of error? What causes it?", "expect": "concept"},
]

## 2. Classifier

The LLM answers in JSON where `route` must be 1 of 3 values (`enum`) — we use **temperature 0** because we want the same answer every time

In [ ]:
# ✏️ TODO 1: Restrict route to only 3 values: concept, code_error, admin
ROUTE_SCHEMA = {
    "type": "object",
    "properties": {"route": {"type": "string", "enum": ___}},
    "required": ["route"], "additionalProperties": False,
}

def classify(item):
    prompt = ("Classify this student question for a university AI course into exactly one route:\n"
              "- concept: asks to explain an idea or theory (no code to debug)\n"
              "- code_error: the student's code fails and they attached it\n"
              "- admin: course logistics such as exam dates, rooms, deadlines, grading policy\n\n"
              # ✏️ TODO 2: Call the LLM with temperature 0 and ROUTE_SCHEMA, then pull out the route value
              f"Question: {item['q']}\nAttached code: {item.get('code', '(none)')}")
    return ___

print(classify(QUESTIONS[0]), classify(QUESTIONS[2]))

## 3. Three specialist handlers

| route | prompt | tool | model |
|---|---|---|---|
| `concept` | explain theory + an analogy | — | Gemma 4 26B |
| `code_error` | read the real error, then explain the fix | `run_snippet()` runs the code | Gemma 4 26B |
| `admin` | answer from the course info only | — | **Gemma 3 4B** (small, cheap) |

In [ ]:
def answer_concept(item):
    return ask_llm("You are a patient teaching assistant for an AI course. Explain the idea clearly, "
                   "with one everyday analogy. Answer in Thai, at most 3 sentences.\n\n"
                   f"Question: {item['q']}")

In [ ]:
import subprocess, sys

def run_snippet(code):
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=10,
                     env={**os.environ, "PYTHON_COLORS": "0"})
    return r.stderr.strip().splitlines()[-1] if r.returncode else "no error"

# ✏️ TODO 3: First run the student's attached code with the tool, then put the real error into the prompt

def answer_code(item):
    error = ___
    return ask_llm("You are a Python debugging tutor. Using the REAL error below, explain the cause "
                   "in Thai in at most 2 sentences, then give one fixed line of code.\n\n"
                   f"Code:\n{item['code']}\n\nError when run: {___}"), error

In [ ]:
SMALL_MODEL = "google/gemma-3-4b-it"
COURSE_INFO = ("Midterm exam: Saturday 12 December, 09:00-12:00, room CL-301. Final exam: Saturday 6 February, room CL-301. "
               "Late lab submission: -10% per day, not accepted after 3 days. Office hours: Wed 13:00-15:00.")

# ✏️ TODO 4: Send admin questions to the small model SMALL_MODEL instead of the main MODEL

def answer_admin(item):
    resp = client.chat.completions.create(model=___, temperature=0, messages=[{"role": "user", "content":
        "Answer the student's question using ONLY this course info, copying dates and rooms exactly. "
      "If it is not there, say to ask the instructor. "
        f"Answer in Thai, 1-2 sentences.\n\nCourse info: {COURSE_INFO}\n\nQuestion: {item['q']}"}])
    return resp.choices[0].message.content

## 4. Router: classify once, then hand off

`question → classify() → ROUTES[route] → answer`

In [ ]:
# ✏️ TODO 5: Build a dict that maps each route name to its handler function
ROUTES = {"concept": ___, "code_error": ___, "admin": ___}

# ✏️ TODO 6: Classify once, then call the handler that matches the label from ROUTES

def route(item):
    label = ___
    return label, ___

In [ ]:
results = []
for item in QUESTIONS:
    label, answer = route(item)
    results.append((item, label, answer))

print(f"{'question':<46} {'expect':<11} {'route':<11} ok  handler")
for item, label, _ in results:
    ok = "✅" if label == item["expect"] else "❌"
    print(f"{item['q'][:44]:<46} {item['expect']:<11} {label:<11} {ok}  {ROUTES[label].__name__}")

In [ ]:
for item, label, answer in results:
    print(f"\n[{label}] {item['q']}")
    if label == "code_error":
        answer, error = answer
        print("  🔧 tool:", error)
    print("  →", answer.strip())

## Summary

| Component | In the code |
|---|---|
| Classifier | `classify()` — structured output, 3-value `enum`, temperature 0 |
| Specialist handlers | `answer_concept` (prompt) · `answer_code` (tool `run_snippet`) · `answer_admin` (small model) |
| Dispatch | `ROUTES[label](item)` — classify once, then hand off |

**Try next:** add a new `off_topic` route (e.g. "Can you recommend a coffee shop near campus?") that politely says it's outside the course scope —
you only need to change the `enum`, the classifier prompt, and `ROUTES`